# 22 · GRPO and its fixes (Dr. GRPO, DAPO), verifiable rewards, pass@k

**Needed for:** DeepSeekMath's GRPO (RL track, step R5), DeepSeek-R1-Zero-style training (step R6), and
DAPO / Dr. GRPO (step R7).

**GRPO** drops PPO's value model. For each prompt, sample a **group** of $G$ responses, score each (often with a
program that checks the answer), and use the group's own statistics as the baseline:
$$A_i = \frac{r_i - \text{mean}(r_{1..G})}{\text{std}(r_{1..G})}$$
Every token of response $i$ gets advantage $A_i$. The loss is PPO's clipped objective per token, plus a k3 KL penalty
to a reference model.

Later papers found biases in the details, which is what this notebook is mostly about:

| Fix | Paper | Change |
|---|---|---|
| No std division | Dr. GRPO | dividing by std over-weights prompts that are nearly always right or wrong |
| No per-response length normalization | Dr. GRPO, DAPO | averaging per response first gives tokens of short responses more weight |
| Clip-higher | DAPO | $\epsilon_{high} = 0.28 > \epsilon_{low} = 0.2$: lets unlikely tokens grow, keeps exploration |
| Dynamic sampling | DAPO | drop groups where all rewards are equal: their advantages are all zero |
| Overlong shaping | DAPO | soft penalty near the length limit instead of a hard cut-off |

In [ ]:
import math
import re
from itertools import combinations
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Group-relative advantages

In [ ]:
def group_advantages(rewards, group_size, scale_by_std=True, eps=1e-6):
    """rewards: (n_prompts * group_size,) ordered group by group. Returns advantages, same shape."""
    r = rewards.view(-1, group_size)
    adv = r - r.mean(1, keepdim=True)
    if scale_by_std:
        adv = adv / (r.std(1, keepdim=True) + eps)
    return adv.flatten()


def test_group_adv(fn):
    r = torch.tensor([1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0])     # two prompts, G = 4
    a = fn(r, 4)
    torch.testing.assert_close(a[4:], torch.zeros(4))              # all correct: no signal
    torch.testing.assert_close(a[:4].sum(), torch.tensor(0.0), atol=1e-5, rtol=0)
    assert a[0] > 0 and (a[1:4] < 0).all()
    torch.testing.assert_close(fn(r, 4, scale_by_std=False)[:4], torch.tensor([0.75, -0.25, -0.25, -0.25]))


test_group_adv(group_advantages)
for n_correct in (1, 4, 7):
    r = torch.tensor([1.0] * n_correct + [0.0] * (8 - n_correct))
    with_std, without = group_advantages(r, 8)[0].item(), group_advantages(r, 8, scale_by_std=False)[0].item()
    print(f"{n_correct}/8 correct: advantage of a correct answer {with_std:.2f} with std scaling, {without:.2f} without")

Dividing by the std blows up the advantages in groups that nearly all agree: a lone success on a hard prompt
(1/8 correct) gets about 2.5 instead of 0.88. So prompts that are almost always wrong, or almost always right,
get outsized updates compared with the prompts in between. That's Dr. GRPO's "question difficulty bias" argument
for dropping the std.

## 2. How the per-token losses are averaged

Given per-token losses $L_{i,t}$ and a response mask:
- **GRPO (original):** average over each response's tokens, then over responses. A token in a 10-token response
  weighs 10× more than one in a 100-token response.
- **DAPO (token-level):** sum over all tokens in the batch ÷ total tokens. Every token weighs the same.
- **Dr. GRPO:** sum ÷ a fixed constant (e.g. max length × number of responses), so the weight doesn't depend on
  the batch's lengths either.

In [ ]:
def aggregate(loss_tok, mask, mode, max_len=None):
    if mode == "seq_mean":
        return ((loss_tok * mask).sum(1) / mask.sum(1)).mean()
    if mode == "token_mean":
        return (loss_tok * mask).sum() / mask.sum()
    if mode == "constant":
        return (loss_tok * mask).sum() / (mask.shape[0] * max_len)
    raise ValueError(mode)


mask = torch.zeros(2, 100)
mask[0, :10] = 1                                  # a 10-token response
mask[1, :100] = 1                                 # a 100-token response
for mode in ("seq_mean", "token_mean", "constant"):
    loss_tok = torch.zeros(2, 100, requires_grad=True)
    aggregate(loss_tok, mask, mode, max_len=100).backward()
    w_short, w_long = loss_tok.grad[0, 0].item(), loss_tok.grad[1, 0].item()
    print(f"{mode:10s} weight per token: short response {w_short:.4f}, long response {w_long:.4f}")

Why it matters: with GRPO's per-response average, a *wrong* long response spreads its negative advantage thinly
over many tokens, so it's penalized less per token than a wrong short one. The model drifts toward long wrong
answers, one reason for the "responses keep getting longer" effect.

## 3. The GRPO / DAPO loss

In [ ]:
def grpo_loss(logp, old_logp, ref_logp, adv, mask, eps_low=0.2, eps_high=0.2, kl_coef=0.04, agg="token_mean"):
    """logp, old_logp, ref_logp, mask: (N, T) per-token; adv: (N,) one advantage per response."""
    ratio = torch.exp(logp - old_logp)
    a = adv[:, None]
    pg = -torch.min(ratio * a, ratio.clamp(1 - eps_low, 1 + eps_high) * a)
    log_r = ref_logp - logp
    kl = torch.expm1(log_r) - log_r                  # k3 estimator, per token
    return aggregate(pg + kl_coef * kl, mask, agg, max_len=mask.shape[1])


def test_grpo_loss(fn):
    torch.manual_seed(0)
    N, T = 4, 5
    old = torch.randn(N, T)
    mask = torch.ones(N, T)
    adv = torch.tensor([1.0, -1.0, 0.5, 0.0])
    # at the start of an update (logp == old == ref), the loss is -mean(adv) over tokens and the KL term is zero
    loss = fn(old.clone(), old, old, adv, mask, kl_coef=0.1)
    torch.testing.assert_close(loss, -adv.mean())
    # clip-higher: a token with A > 0 and ratio 1.25 is still inside (1 + 0.28), so it keeps a gradient
    logp = (old + math.log(1.25)).requires_grad_(True)
    fn(logp, old, old, torch.ones(N), mask, eps_high=0.28, kl_coef=0.0).backward()
    assert (logp.grad != 0).all()
    logp.grad = None
    fn(logp, old, old, torch.ones(N), mask, eps_high=0.2, kl_coef=0.0).backward()
    assert (logp.grad == 0).all()


test_grpo_loss(grpo_loss)
print("GRPO loss ok (with clip-higher)")

## 4. Verifiable rewards and DAPO's overlong shaping

For math, the reward is a program: extract the final answer and compare. Keep it strict and simple. Models find
every loophole in a lenient parser (reward hacking).

In [ ]:
def extract_answer(text):
    boxed = re.findall(r"\\boxed\{([^{}]*)\}", text)
    if boxed:
        return boxed[-1].strip()
    hashes = re.findall(r"####\s*([^\n]+)", text)
    return hashes[-1].strip() if hashes else None


def correctness_reward(text, gold):
    ans = extract_answer(text)
    if ans is None:
        return 0.0
    try:
        return float(abs(float(ans.replace(",", "")) - float(gold)) < 1e-6)
    except ValueError:
        return float(ans == str(gold))


def overlong_penalty(length, max_len=100, buffer=20):
    """DAPO: 0 until max_len - buffer, then linearly down to -1 at max_len, -1 beyond."""
    if length <= max_len - buffer:
        return 0.0
    if length <= max_len:
        return (max_len - buffer - length) / buffer
    return -1.0


def test_reward(fn):
    assert fn("so the answer is \\boxed{42}", 42) == 1.0
    assert fn("first \\boxed{41} then actually \\boxed{42}", 42) == 1.0    # last boxed answer counts
    assert fn("... #### 1,234", 1234) == 1.0
    assert fn("the answer is 42", 42) == 0.0                              # no final-answer format: no reward
    assert fn("\\boxed{forty-two}", 42) == 0.0


test_reward(correctness_reward)
print([overlong_penalty(n) for n in (50, 80, 90, 100, 120)])

## 5. pass@k, the unbiased estimator

Sample $n \ge k$ responses, count $c$ correct. The probability that at least one of $k$ randomly chosen samples is
correct is $1 - \binom{n-c}{k}/\binom{n}{k}$. Computing it as a product avoids huge binomials. (Taking $k$ samples
once and checking is much noisier; always sample $n > k$.)

In [ ]:
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - math.prod(1 - k / i for i in range(n - c + 1, n + 1))


def test_pass_at_k(fn):
    for n, c, k in ((5, 2, 2), (10, 3, 4), (8, 0, 3), (6, 6, 2), (7, 1, 1)):
        outcomes = [1] * c + [0] * (n - c)
        brute = sum(any(outcomes[i] for i in subset) for subset in combinations(range(n), k)) / math.comb(n, k)
        assert math.isclose(fn(n, c, k), brute), (n, c, k)


test_pass_at_k(pass_at_k)
print(f"3 of 10 samples correct: pass@1 = {pass_at_k(10, 3, 1):.3f}, pass@5 = {pass_at_k(10, 3, 5):.3f}")

## 6. GRPO on a toy task, and entropy collapse

The "prompt" is two digits $a, b$; the "response" is one token, the sum $0 \dots 18$. The policy is an MLP.
Each step: 8 samples per prompt, reward 1 if correct, group advantages without std, two epochs of the clipped loss.

**Dynamic sampling** (DAPO) drops groups where every sample got the same reward, then samples more prompts until
64 useful groups are collected (at most 8 tries here).

Watch the policy's **entropy**. When it collapses, all 8 samples of a prompt come out identical, right or wrong,
so no group has any signal and learning stalls on the prompts it still gets wrong. The second run adds an entropy
bonus to the loss.

In [ ]:
def encode(a, b):
    return torch.cat([F.one_hot(a, 10), F.one_hot(b, 10)], -1).float()


def collect(policy, G=8, want=64, max_tries=8):
    """Dynamic sampling: keep only groups with mixed rewards, refill until `want` groups or tries run out."""
    xs, acts, olds, advs, total_reward, tries = [], [], [], [], [], 0
    while sum(len(t) for t in acts) < want * G and tries < max_tries:
        tries += 1
        a, b = torch.randint(0, 10, (want,)), torch.randint(0, 10, (want,))
        x = encode(a, b).repeat_interleave(G, 0)
        with torch.no_grad():
            dist = torch.distributions.Categorical(logits=policy(x))
            act = dist.sample()
            old = dist.log_prob(act)
        r = (act == (a + b).repeat_interleave(G, 0)).float()
        total_reward.append(r.mean().item())
        keep = r.view(-1, G).std(1).gt(0).repeat_interleave(G)
        xs.append(x[keep]); acts.append(act[keep]); olds.append(old[keep])
        advs.append(group_advantages(r, G, scale_by_std=False)[keep])
    cat = lambda t: torch.cat(t)[: want * G]
    return cat(xs), cat(acts), cat(olds), cat(advs), sum(total_reward) / len(total_reward), tries


def run_grpo(ent_coef, steps=301, seed=0):
    torch.manual_seed(seed)
    policy = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 19))
    opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
    print(f"entropy bonus = {ent_coef}")
    for step in range(steps):
        x, act, old, adv, reward, tries = collect(policy)
        if len(act) > 0:
            for _ in range(2):
                dist = torch.distributions.Categorical(logits=policy(x))
                logp = dist.log_prob(act)
                loss = grpo_loss(logp[:, None], old[:, None], old[:, None], adv, torch.ones(len(act), 1),
                                 eps_high=0.28, kl_coef=0.0) - ent_coef * dist.entropy().mean()
                opt.zero_grad()
                loss.backward()
                opt.step()
        if step % 60 == 0:
            with torch.no_grad():
                ta, tb = torch.randint(0, 10, (2000,)), torch.randint(0, 10, (2000,))
                logits = policy(encode(ta, tb))
                acc = (logits.argmax(-1) == ta + tb).float().mean().item()
                ent = torch.distributions.Categorical(logits=logits).entropy().mean().item()
            print(f"  step {step:3d}  sample reward {reward:.2f}  useful groups {len(act) // 8:2d}/64 after {tries} tries  "
                  f"entropy {ent:.2f}  greedy accuracy {acc:.2f}")


run_grpo(ent_coef=0.0)
run_grpo(ent_coef=0.05)

What to notice:
- Both runs stall short of 100% once entropy falls to about 0.1 and useful groups run out. The loop keeps going
  but has almost nothing to learn from. That's entropy collapse.
- The entropy bonus learns faster and ends a few points higher, but doesn't prevent the collapse at this
  learning rate.
- Clip-higher (used in both runs) can't matter much here: with one-token responses and 2 epochs, ratios rarely
  reach the clip boundary. It matters in real runs with many tokens and more reuse of each batch.

Try it: lower the learning rate, raise G to 16, or raise the entropy bonus, and rerun with a few seeds. Toy
results vary, so compare across seeds before concluding anything.

## Things to be able to explain

- GRPO with one gradient step per batch (fully on-policy): what does the clipping do? (Nothing; the ratio is 1.)
- Why is a KL penalty often dropped for reasoning RL (DAPO) but kept for chat RLHF?
- Why does training throughput depend on the *longest* response in a batch, and what do systems do about it?
  (Rollout generation dominates; notebook 23.)

## Drill

In [ ]:
def my_group_advantages(rewards, group_size, scale_by_std=True, eps=1e-6):
    raise NotImplementedError


def my_grpo_loss(logp, old_logp, ref_logp, adv, mask, eps_low=0.2, eps_high=0.2, kl_coef=0.04, agg="token_mean"):
    raise NotImplementedError


def my_pass_at_k(n, c, k):
    raise NotImplementedError


drill(test_group_adv, my_group_advantages)
drill(test_grpo_loss, my_grpo_loss)
drill(test_pass_at_k, my_pass_at_k)